# Process Model Matching on PMMC 2015: Graph Context for LLM Matching

Does an LLM match process activities better when it also sees each activity's neighbourhood in the process graph? This follows the Graph-RAG setting of Zarrinkia, Srinivasan and Thomo (2026), *The Reasoning Bottleneck in Graph-RAG*, applied to activity matching.

All methods are scored on the same candidate pairs produced by `01_data_and_baselines.ipynb`:

| Method | Input to the model |
|---|---|
| Labels only | the two activity labels |
| Graph context | labels + up to 3 preceding and 3 following activities + actor |
| Graph context + checklist | as above, with a short structured reasoning checklist |
| Hybrid | Gradient Boosting on the baseline features plus the three LLM scores |

Input: `outputs.zip` from notebook 01 (upload it to the session before running). Runtime: GPU (a Colab T4 is sufficient).

## Setup

In [ ]:
!pip -q install -U transformers accelerate bitsandbytes networkx scikit-learn pandas tqdm

In [ ]:
import os, re, json, zipfile, random
import numpy as np, pandas as pd, networkx as nx
from tqdm.auto import tqdm
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import GroupKFold
from sklearn.metrics import precision_recall_fscore_support

SEED = 42; random.seed(SEED); np.random.seed(SEED)

assert os.path.exists("outputs.zip"), "outputs.zip from notebook 01 is required."
with zipfile.ZipFile("outputs.zip") as z:
    z.extractall(".")
IN_DIR, OUT_DIR = "outputs", "outputs_graph_context"
os.makedirs(OUT_DIR, exist_ok=True)

pairs = pd.read_csv(f"{IN_DIR}/pairs.csv", dtype={"id1": str, "id2": str})
baselines = pd.read_csv(f"{IN_DIR}/results_baselines.csv")
graphs = {}
for f in os.listdir(IN_DIR):
    if f.endswith(".graphml"):
        graphs[f[:-8]] = nx.read_graphml(f"{IN_DIR}/{f}")

print(f"{len(graphs)} process graphs | {len(pairs):,} pairs | {int(pairs.y.sum())} gold matches | "
      f"{int(pairs.candidate.sum()):,} candidates")
baselines

## Graph context

For each activity, retrieve its preceding activities, following activities and actor. Context is capped at three neighbours in each direction to keep prompts short.

In [ ]:
def neighbours(G, node, direction, max_n=3):
    nbrs = G.predecessors(node) if direction == "before" else G.successors(node)
    return [G.nodes[n].get("label", "") for n in list(nbrs)[:max_n]]

def context_block(model, node, name):
    G = graphs[model]; d = G.nodes[node]
    before = neighbours(G, node, "before") or ["(process start)"]
    after = neighbours(G, node, "after") or ["(process end)"]
    return (f"{name}: \"{d.get('label', '')}\"\n"
            f"  performed by: {d.get('actor', 'unknown')}\n"
            f"  comes after: {'; '.join(before)}\n"
            f"  followed by: {'; '.join(after)}")

r = pairs[pairs.y == 1].iloc[0]
print(context_block(r.m1, r.id1, "Activity A")); print()
print(context_block(r.m2, r.id2, "Activity B"))

## Model

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

LLM_NAME = "Qwen/Qwen2.5-7B-Instruct"   # use the same model as notebook 01
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16, bnb_4bit_quant_type="nf4")
tok = AutoTokenizer.from_pretrained(LLM_NAME); tok.padding_side = "left"
llm = AutoModelForCausalLM.from_pretrained(LLM_NAME, quantization_config=bnb, device_map="auto").eval()

YES = [tok.encode(w, add_special_tokens=False)[0] for w in ["Yes", " Yes", "yes"]]
NO  = [tok.encode(w, add_special_tokens=False)[0] for w in ["No", " No", "no"]]

SYSTEM = ("You are an expert in business process management. Two universities describe their "
          "admission processes. Decide if two activities mean the same thing.")

def chat(user):
    return tok.apply_chat_template([{"role": "system", "content": SYSTEM}, {"role": "user", "content": user}],
                                   tokenize=False, add_generation_prompt=True)

@torch.no_grad()
def p_yes(prompts, bs=8):
    out = []
    for i in tqdm(range(0, len(prompts), bs), desc="LLM"):
        enc = tok(prompts[i:i+bs], return_tensors="pt", padding=True).to(llm.device)
        probs = torch.softmax(llm(**enc).logits[:, -1, :].float(), -1)
        y, n = probs[:, YES].sum(-1), probs[:, NO].sum(-1)
        out.extend((y / (y + n + 1e-9)).cpu().tolist())
    return np.array(out)

## Prompts

In [ ]:
def prompt_graph(r):
    return chat(f"{context_block(r.m1, r.id1, 'Activity A (university 1)')}\n\n"
                f"{context_block(r.m2, r.id2, 'Activity B (university 2)')}\n\n"
                "Do A and B describe the same activity? Answer only Yes or No.")

def prompt_structured(r):
    return chat(f"{context_block(r.m1, r.id1, 'Activity A (university 1)')}\n\n"
                f"{context_block(r.m2, r.id2, 'Activity B (university 2)')}\n\n"
                "Check step by step:\n"
                "1. Do the two labels describe the same action on the same object (synonyms count)?\n"
                "2. Is the activity performed by the same kind of actor?\n"
                "3. Do the steps before and after play the same role in both processes?\n"
                "Two activities are the same only if check 1 holds and checks 2-3 do not contradict it.\n"
                "Do A and B describe the same activity? Answer only Yes or No.")

cand = pairs[pairs.candidate].copy()
print(prompt_structured(next(cand.itertuples())))

## Scoring

In [ ]:
cand["llm_graph"] = p_yes([prompt_graph(r) for r in cand.itertuples()])
cand["llm_struct"] = p_yes([prompt_structured(r) for r in cand.itertuples()])
for c in ["llm_graph", "llm_struct"]:
    pairs[c] = 0.0
    pairs.loc[cand.index, c] = cand[c]
pairs.to_csv(f"{OUT_DIR}/pairs_graph_context.csv", index=False)   # checkpoint

## Evaluation

Each LLM score is evaluated with a fixed threshold (P(Yes) ≥ 0.5) and with a threshold tuned by cross-validation grouped by model pair. Non-candidate pairs count as negatives, so every method is scored on all pairs.

In [ ]:
def prf(y, p):
    P, R, F, _ = precision_recall_fscore_support(y, p, average="binary", zero_division=0)
    return round(P, 3), round(R, 3), round(F, 3)

def best_threshold(y, s):
    return max(np.linspace(0.05, 0.95, 91), key=lambda t: prf(y, (s >= t).astype(int))[2])

gkf = GroupKFold(n_splits=6)
def tuned_pred(score):
    pred = np.zeros(len(pairs), dtype=int)
    for tr, te in gkf.split(pairs, pairs.y, pairs.group):
        t = best_threshold(pairs.y.values[tr], score[tr])
        pred[te] = (score[te] >= t).astype(int)
    return pred

y = pairs.y.values
rows = []
for name, col in [("LLM, labels only", "llm_label_only"),
                  ("LLM + graph context", "llm_graph"),
                  ("LLM + graph context + checklist", "llm_struct")]:
    s = pairs[col].values
    rows.append((name + " (threshold 0.5)", *prf(y, (s >= 0.5).astype(int))))
    rows.append((name + " (tuned threshold)", *prf(y, tuned_pred(s))))

# Hybrid: baseline features plus LLM scores
FEATS = ["seq_ratio", "jaccard", "cos_label", "cos_desc", "same_actor"]
def gb_cv(feats):
    pred = np.zeros(len(pairs), dtype=int); score = np.zeros(len(pairs))
    for tr, te in gkf.split(pairs, pairs.y, pairs.group):
        ytr = y[tr]
        w = np.where(ytr == 1, (ytr == 0).sum() / max(1, ytr.sum()), 1.0)
        clf = GradientBoostingClassifier(random_state=SEED).fit(pairs[feats].values[tr], ytr, sample_weight=w)
        t = best_threshold(ytr, clf.predict_proba(pairs[feats].values[tr])[:, 1])
        score[te] = clf.predict_proba(pairs[feats].values[te])[:, 1]
        pred[te] = (score[te] >= t).astype(int)
    return pred, score

pred_b1, _ = gb_cv(FEATS)
rows.append(("Feature-based (T5 + Gradient Boosting)", *prf(y, pred_b1)))
pred_m5, score_m5 = gb_cv(FEATS + ["llm_label_only", "llm_graph", "llm_struct"])
pairs["m5_score"] = score_m5; pairs["m5_pred"] = pred_m5
rows.append(("Hybrid (features + LLM scores)", *prf(y, pred_m5)))

results = pd.DataFrame(rows, columns=["Method", "Precision", "Recall", "F1"])
results.to_csv(f"{OUT_DIR}/results_graph_context.csv", index=False)
results

## Decisions changed by graph context

Candidate pairs where the label-only and checklist decisions disagree, split into corrections and new errors.

In [ ]:
c = pairs[pairs.candidate].copy()
c["label_only_says"] = c.llm_label_only >= 0.5
c["graph_says"] = c.llm_struct >= 0.5
fixed = c[(c.label_only_says != c.graph_says) & (c.graph_says == c.y.astype(bool))]
broke = c[(c.label_only_says != c.graph_says) & (c.graph_says != c.y.astype(bool))]
print(f"Corrected: {len(fixed)} | new errors: {len(broke)}\n")
for r in fixed.head(4).itertuples():
    print("match" if r.y else "no match", f"| P(Yes) labels only {r.llm_label_only:.2f} -> with context {r.llm_struct:.2f}")
    print(context_block(r.m1, r.id1, "A")); print(context_block(r.m2, r.id2, "B")); print("-" * 70)

## Qualitative examples

The model writes out each checklist step before its decision, for three of the corrected pairs.

In [ ]:
@torch.no_grad()
def explain(r, max_new_tokens=200):
    p = prompt_structured(r).replace("Answer only Yes or No.",
                                     "Write one short line per check, then 'Final answer: Yes' or 'Final answer: No'.")
    enc = tok([p], return_tensors="pt").to(llm.device)
    out = llm.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False)
    return tok.decode(out[0][enc["input_ids"].shape[1]:], skip_special_tokens=True)

for r in (fixed if len(fixed) else c[c.y == 1]).head(3).itertuples():
    print(context_block(r.m1, r.id1, "A")); print(context_block(r.m2, r.id2, "B"))
    print("Gold:", "match" if r.y else "no match"); print(explain(r)); print("=" * 70)

## Outputs

In [ ]:
pairs.to_csv(f"{OUT_DIR}/pairs_graph_context.csv", index=False)
all_results = pd.concat([baselines.assign(notebook="01"), results.assign(notebook="02")], ignore_index=True)
all_results.to_csv(f"{OUT_DIR}/all_results.csv", index=False)
!zip -qr outputs_graph_context.zip {OUT_DIR}
print("Saved outputs_graph_context/ and outputs_graph_context.zip")
results